# Wikipedia Redirect Index Demo

这个 notebook 用来加载、检查和测试 `WikipediaRedirectIndex`。

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Working directory: {REPO_ROOT}")


In [1]:
from dataclasses import asdict

from pathlib import Path

from wikipedia_redirects import (
    WikipediaRedirectIndex,
    get_bucket_key,
    normalize_wikipedia_title,
)

INDEX_DIR = Path("data/wikipedia_redirects/redirect_index")
INDEX_DIR

PosixPath('data/wikipedia_redirects/redirect_index')

In [2]:
if not INDEX_DIR.exists():
    raise FileNotFoundError(f"Index directory not found: {INDEX_DIR}")

with WikipediaRedirectIndex(INDEX_DIR) as index:
    stats = index.stats()
    graph_stats = index.graph_stats()
    metadata = {
        "wiki": index.get_metadata("wiki"),
        "dump_tag": index.get_metadata("dump_tag"),
        "page_dump": index.get_metadata("page_dump"),
        "redirect_dump": index.get_metadata("redirect_dump"),
    }

{
    "stats": stats,
    "graph_stats": graph_stats,
    "metadata": metadata,
}

{'stats': {'canonical_pages': 2629590, 'redirects': 8429635},
 'graph_stats': {'redirect_nodes': 8429635,
  'target_nodes': 2629590,
  'max_total_nodes': 11059225,
  'directed_edges': 8429635,
  'undirected_edges': 8429635},
 'metadata': {'wiki': 'enwiki',
  'dump_tag': 'latest',
  'page_dump': 'data/wikipedia_redirects/raw/enwiki-latest-page.sql.gz',
  'redirect_dump': 'data/wikipedia_redirects/raw/enwiki-latest-redirect.sql.gz'}}

## 1. 单个标题查询

In [3]:
title = "USA"

normalized = normalize_wikipedia_title(title)
bucket_key = get_bucket_key(normalized)

with WikipediaRedirectIndex(INDEX_DIR) as index:
    canonical = index.resolve_redirect(title)
    synonyms = index.get_filtered_synonyms(title)

{
    "title": title,
    "normalized": normalized,
    "bucket_key": bucket_key,
    "canonical": canonical,
    "synonym_count": len(synonyms),
    "synonyms": synonyms,
}

{'title': 'USA',
 'normalized': 'usa',
 'bucket_key': 'us',
 'canonical': 'united states',
 'synonym_count': 21,
 'synonyms': ['united states',
  'american united states',
  'federal united states',
  'inited states',
  'nited states',
  'states united',
  'the united states',
  'unietd states',
  'unitd states',
  'unite states',
  'united american states',
  'united satates',
  'united sates',
  'united staes',
  'united stated',
  'united statees',
  'united states america',
  'united statez',
  'unites states',
  'untied states',
  'vnited states']}

## 2. 批量测试多个查询词

In [4]:
queries = [
    "USA",
    "U.S.A.",
    "US",
    "United States",
    "NYC",
    "New York City",
    "UK",
    "United Kingdom",
]

rows = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for query in queries:
        rows.append({
            "query": query,
            "normalized": normalize_wikipedia_title(query),
            "bucket": get_bucket_key(normalize_wikipedia_title(query)),
            "canonical": index.resolve_redirect(query),
            "synonym_count": len(index.get_synonyms(query)),
        })

rows

[{'query': 'USA',
  'normalized': 'usa',
  'bucket': 'us',
  'canonical': 'united states',
  'synonym_count': 171},
 {'query': 'U.S.A.',
  'normalized': 'u s a',
  'bucket': 'us',
  'canonical': 'united states',
  'synonym_count': 171},
 {'query': 'US',
  'normalized': 'us',
  'bucket': 'us',
  'canonical': 'chicago med season 1',
  'synonym_count': 16},
 {'query': 'United States',
  'normalized': 'united states',
  'bucket': 'un',
  'canonical': 'american wine',
  'synonym_count': 15},
 {'query': 'NYC',
  'normalized': 'nyc',
  'bucket': 'ny',
  'canonical': 'new york city',
  'synonym_count': 65},
 {'query': 'New York City',
  'normalized': 'new york city',
  'bucket': 'ne',
  'canonical': 'the men',
  'synonym_count': 11},
 {'query': 'UK',
  'normalized': 'uk',
  'bucket': 'uk',
  'canonical': 'ukca marking',
  'synonym_count': 6},
 {'query': 'United Kingdom',
  'normalized': 'united kingdom',
  'bucket': 'un',
  'canonical': 'kingdom of great britain',
  'synonym_count': 16}]

## 3. 查看某个 canonical 的同义词

In [ ]:
canonical_title = "United States"

with WikipediaRedirectIndex(INDEX_DIR) as index:
    synonyms = index.get_synonyms(canonical_title)

print(f"Canonical: {canonical_title}")
print(f"Total synonyms: {len(synonyms)}")
synonyms[:100]

## 4. 直接查看 bucket 文件内容

In [ ]:
import gzip
import pickle

bucket_to_inspect = "us"
redirect_bucket_path = INDEX_DIR / "redirect_buckets" / f"{bucket_to_inspect}.pkl.gz"
canonical_bucket_path = INDEX_DIR / "canonical_buckets" / f"{bucket_to_inspect}.pkl.gz"

def load_pickle_gz(path: Path):
    with gzip.open(path, "rb") as f:
        return pickle.load(f)

redirect_bucket = load_pickle_gz(redirect_bucket_path) if redirect_bucket_path.exists() else {}
canonical_bucket = load_pickle_gz(canonical_bucket_path) if canonical_bucket_path.exists() else {}

print("redirect bucket size:", len(redirect_bucket))
print("canonical bucket size:", len(canonical_bucket))

In [ ]:
list(redirect_bucket.items())[:20]

In [ ]:
list(canonical_bucket.items())[:10]

## 5. 遍历前几个 redirect pair

In [ ]:
pairs = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for idx, pair in enumerate(index.iter_pairs()):
        pairs.append(pair)
        if idx >= 19:
            break

pairs

## 6. 写一个便于反复调用的小函数

In [ ]:
def inspect_title(title: str, synonym_limit: int = 20):
    normalized = normalize_wikipedia_title(title)
    bucket_key = get_bucket_key(normalized)
    with WikipediaRedirectIndex(INDEX_DIR) as index:
        canonical = index.resolve_redirect(title)
        synonyms = index.get_synonyms(title)
    return {
        "title": title,
        "normalized": normalized,
        "bucket_key": bucket_key,
        "canonical": canonical,
        "synonym_count": len(synonyms),
        "synonyms": synonyms[:synonym_limit],
    }

inspect_title("USA")

## 7. 查看图节点和邻居

In [ ]:
node_query = "USA"

with WikipediaRedirectIndex(INDEX_DIR) as index:
    node = index.get_node(node_query)
    neighbors = index.get_neighbors(node_query)

{
    "query": node_query,
    "node": asdict(node) if node else None,
    "neighbor_count": len(neighbors),
    "neighbors": neighbors[:20],
}

## 8. 测试 n-hop 连通和最短路径

In [ ]:
examples = [
    ("USA", "United States", 1),
    ("USA", "United States", 0),
    ("USA", "US", 3),
    ("USA", "US", 2),
    ("United States", "American wine", 1),
    ("USA", "United Kingdom", 5),
]

rows = []
with WikipediaRedirectIndex(INDEX_DIR) as index:
    for source_title, target_title, max_hops in examples:
        rows.append({
            "source": source_title,
            "target": target_title,
            "max_hops": max_hops,
            "distance": index.hop_distance(source_title, target_title),
            "within_n_hops": index.are_connected_within_hops(source_title, target_title, max_hops),
            "shortest_path": index.shortest_path(source_title, target_title),
        })

rows

## 9. 写一个便于反复测试图关系的小函数

In [ ]:
def inspect_relation(source_title: str, target_title: str, max_hops: int = 2, directed: bool = False):
    with WikipediaRedirectIndex(INDEX_DIR) as index:
        source_node = index.get_node(source_title)
        target_node = index.get_node(target_title)
        return {
            "source_node": asdict(source_node) if source_node else None,
            "target_node": asdict(target_node) if target_node else None,
            "directed": directed,
            "max_hops": max_hops,
            "distance": index.hop_distance(source_title, target_title, directed=directed),
            "within_n_hops": index.are_connected_within_hops(source_title, target_title, max_hops, directed=directed),
            "path": index.shortest_path(source_title, target_title, directed=directed),
        }

inspect_relation("USA", "U.S.A.", max_hops=2)